# Cloning repos

This will clone all the repos from /projects, the cloning is required to extract the commit data.

In [ ]:
import os
import subprocess

ROOT_FOLDER = r"../projects"
FULL_REPO_ROOT = os.path.join(ROOT_FOLDER, "_full_repositories")
os.makedirs(FULL_REPO_ROOT, exist_ok=True)

def run_git(repo_path, args):
    result = subprocess.run(
        ["git"] + args,
        cwd=repo_path,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed:\n{result.stderr.strip()}"
        )

    return result.stdout.strip()

repositories = {}

for name in sorted(os.listdir(ROOT_FOLDER)):
    snapshot_path = os.path.join(ROOT_FOLDER, name)

    if not os.path.isdir(snapshot_path):
        continue

    if name.startswith(".") or name == "_full_repositories":
        continue

    git_path = os.path.join(snapshot_path, ".git")

    if not os.path.isdir(git_path):
        continue

    try:
        origin_url = run_git(
            snapshot_path,
            ["config", "--get", "remote.origin.url"]
        )

        if origin_url:
            repositories[origin_url] = True

    except Exception as e:
        print(f"Could not read remote from {name}: {e}")

print(f"Found {len(repositories)} unique repositories.")

downloaded = []
skipped = []
failed = []

for origin_url in sorted(repositories):
    url = origin_url.rstrip("/")

    if url.endswith(".git"):
        url = url[:-4]

    if "github.com/" in url:
        repo_name = url.split("github.com/", 1)[1]
    elif "github.com:" in url:
        repo_name = url.split("github.com:", 1)[1]
    else:
        repo_name = os.path.basename(url)

    repo_dir_name = repo_name.replace("/", "_")
    repo_path = os.path.join(FULL_REPO_ROOT, repo_dir_name)

    print()
    print("=" * 80)
    print(f"Repository: {origin_url}")
    print(f"Local path: {repo_path}")

    if os.path.isdir(os.path.join(repo_path, ".git")):
        try:
            commit_count = run_git(
                repo_path,
                ["rev-list", "--all", "--count"]
            )

            print(f"ALREADY DOWNLOADED — skipping ({commit_count} commits)")
            skipped.append(origin_url)

        except Exception as e:
            print(f"Existing repository appears invalid: {e}")
            failed.append({
                "repository": origin_url,
                "error": str(e)
            })

        continue

    try:
        print("Cloning...")

        result = subprocess.run(
            [
                "git",
                "clone",
                "--no-single-branch",
                origin_url,
                repo_path
            ],
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace"
        )

        if result.returncode != 0:
            import shutil
            if os.path.isdir(repo_path):
                shutil.rmtree(repo_path, ignore_errors=True)
            raise RuntimeError(result.stderr.strip())

        commit_count = run_git(
            repo_path,
            ["rev-list", "--all", "--count"]
        )

        print(f"SUCCESS — {commit_count} commits available")
        downloaded.append(origin_url)

    except Exception as e:
        print()
        print("!!! FAILED !!!")
        print(str(e))

        failed.append({
            "repository": origin_url,
            "error": str(e)
        })

print()
print("=" * 80)
print("CLONING COMPLETE")
print("=" * 80)
print(f"Newly downloaded: {len(downloaded)}")
print(f"Already downloaded: {len(skipped)}")
print(f"Failed:            {len(failed)}")

if failed:
    print()
    print("FAILED REPOSITORIES:")

    for item in failed:
        print()
        print(item["repository"])
        print(item["error"])

Found 49 unique repositories.

Repository: https://github.com/AdamaJava/adamajava
Local path: ../projects\_full_repositories\AdamaJava_adamajava
ALREADY DOWNLOADED — skipping (3903 commits)

Repository: https://github.com/Apicurio/apicurio-registry
Local path: ../projects\_full_repositories\Apicurio_apicurio-registry
ALREADY DOWNLOADED — skipping (7787 commits)

Repository: https://github.com/AtsushiSakai/PythonRobotics
Local path: ../projects\_full_repositories\AtsushiSakai_PythonRobotics
ALREADY DOWNLOADED — skipping (2385 commits)

Repository: https://github.com/Azure-Samples/modern-data-warehouse-dataops
Local path: ../projects\_full_repositories\Azure-Samples_modern-data-warehouse-dataops
ALREADY DOWNLOADED — skipping (1016 commits)

Repository: https://github.com/EsupPortail/Esup-Pod
Local path: ../projects\_full_repositories\EsupPortail_Esup-Pod
Cloning...
SUCCESS — 6127 commits available

Repository: https://github.com/Exa-Networks/exabgp
Local path: ../projects\_full_repositor